In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [1]:
# ================================================================
# ✅ CodeRefactor 내부 주요 함수 4개 테스트
# ================================================================

from AutoHPO.ContextManager.ContextManager import (
    get_code_anaylsis_prompt,
    get_param_list_prompt,
    get_code_refactoring_prompt,
    get_code_excute_api_prompt
)
from AutoHPO.Refactor.refactor import CodeRefactor
from pathlib import Path

# ---------------------------------------------------------------
# 0️⃣ 테스트용 입력 코드 준비
# ---------------------------------------------------------------
test_source_code = """
import argparse
import torch
from transformers import AutoModelForSequenceClassification

parser = argparse.ArgumentParser()
parser.add_argument("--epochs", type=int, default=5)
parser.add_argument("--batch_size", type=int, default=16)
parser.add_argument("--lr", type=float, default=3e-5)
parser.add_argument("--save_path", type=str, default="./outputs")

model = AutoModelForSequenceClassification.from_pretrained("klue/roberta-base")
"""

# ---------------------------------------------------------------
# 1️⃣ CodeRefactor 인스턴스 생성
# ---------------------------------------------------------------
refactor = CodeRefactor()
user_requirements = "이 코드는 파인튜닝용 코드야. epochs와 batch_size는 외부에서 조절할 수 있게 해줘."
save_path = Path("./test_target.py")
python_env = Path("/usr/bin/python")
conda_env = "AI"

# ---------------------------------------------------------------
# 2️⃣ _analysis_code() 테스트
# ---------------------------------------------------------------
analysis_prompt = get_code_anaylsis_prompt(
    source_code=test_source_code,
    user_requirements=user_requirements
)
print("=== _analysis_code() 테스트 시작 ===")
is_ml = refactor._analysis_code(prompt=analysis_prompt)
print(f"결과: {is_ml}\n")

# ---------------------------------------------------------------
# 3️⃣ _get_param_list() 테스트
# ---------------------------------------------------------------
param_prompt = get_param_list_prompt(
    source_code=test_source_code,
    user_requirements=user_requirements
)
print("=== _get_param_list() 테스트 시작 ===")
param_list = refactor._get_param_list(prompt=param_prompt)
print(f"결과: {param_list}\n")

# ---------------------------------------------------------------
# 4️⃣ _rewrite_code() 테스트
# ---------------------------------------------------------------
rewrite_prompt = get_code_refactoring_prompt(
    source_code=test_source_code,
    hyper_params=param_list,
    user_requirements=user_requirements,
    save_path=save_path,
    conda_env=conda_env,
    python_env=python_env
)
print("=== _rewrite_code() 테스트 시작 ===")
refactored_code = refactor._rewrite_code(prompt=rewrite_prompt)
print(f"리팩토링된 코드 일부 미리보기:\n{refactored_code[:500]}\n")

# ---------------------------------------------------------------
# 5️⃣ _get_excute_api() 테스트
# ---------------------------------------------------------------
excute_api_prompt = get_code_excute_api_prompt(refactored_code=refactored_code)
print("=== _get_excute_api() 테스트 시작 ===")
api = refactor._get_excute_api(prompt=excute_api_prompt)
print(f"결과: {api}\n")

print("✅ 모든 테스트 단계 완료")


=== _analysis_code() 테스트 시작 ===
결과: True

=== _get_param_list() 테스트 시작 ===
결과: ['epochs', 'batch_size', 'lr']

=== _rewrite_code() 테스트 시작 ===
리팩토링된 코드 일부 미리보기:
import sys, json, time, os
from pathlib import Path
import datetime
import argparse
import torch
from transformers import AutoModelForSequenceClassification

def _save_model_generic(model, path: Path) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        torch.save(model.state_dict(), path)
    except Exception:
        with open(path, "wb") as f:
            pickle.dump(model, f)
    return str(path)

def _emit_json_line(payload: dict) -> None:
    sys.stdout.write(json.du

=== _get_excute_api() 테스트 시작 ===
결과: {'epochs': 1, 'batch_size': 1, 'save_dir': './outputs', 'train_path': './data/train.csv', 'test_path': '', 'healthcheck': 0}

✅ 모든 테스트 단계 완료


In [ ]:
from AutoHPO.Refactor.refactor import CodeRefactor
from pathlib import Path

refactor = CodeRefactor()
reps = refactor.invoke(
    user_requirements=" batch_size=64로 고정할것",
    source_code_path=Path("/home/jeongyuseong/바탕화면/private/오픈소스경진대회/AutoFineTuner_2/target.py"),
    conda_env="AI",
    python_env=Path("/usr/bin/python"),
    save_path=Path("/home/jeongyuseong/바탕화면/private/오픈소스경진대회/AutoFineTuner_2/output")
)

NameError: name 'Path' is not defined

In [20]:
# ✅ 수정된 버전
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field   # ✅ 여기 수정
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

class CodeResponse(BaseModel):
    description: str = Field(..., description="간단한 설명 또는 요약")
    code: str = Field(..., description="실행 가능한 파이썬 코드")
    numList: list[int] = Field(..., description="선택된 숫자 리스트")

parser = PydanticOutputParser(pydantic_object=CodeResponse)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a precise code generator. Output strictly follows the provided JSON schema."),
    ("human", "Write Python code to print 'hello' and pick 3 numbers between 1 and 5."),
    ("human", "{format_instructions}")
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser

result = chain.invoke({})
print(result)
print(result.dict())


description="Prints 'hello' and selects 3 numbers between 1 and 5." code="import random\n\nprint('hello')\nnumbers = random.sample(range(1, 6), 3)\nprint(numbers)" numList=[1, 2, 3]
{'description': "Prints 'hello' and selects 3 numbers between 1 and 5.", 'code': "import random\n\nprint('hello')\nnumbers = random.sample(range(1, 6), 3)\nprint(numbers)", 'numList': [1, 2, 3]}


/tmp/ipykernel_1537635/2135675939.py:26: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  print(result.dict())


In [21]:
result

CodeResponse(description="Prints 'hello' and selects 3 numbers between 1 and 5.", code="import random\n\nprint('hello')\nnumbers = random.sample(range(1, 6), 3)\nprint(numbers)", numList=[1, 2, 3])

In [27]:
print(result.model_dump()["code"])

import random

print('hello')
numbers = random.sample(range(1, 6), 3)
print(numbers)


In [23]:
# from langchain_core.tools import BaseTool, tool


# @tool
# def add(a: int, b: int) -> int:
#     """Adds a and b."""
#     return a + b


# @tool
# def multiply(a: int, b: int) -> int:
#     """Multiplies a and b."""
#     return a * b


# tools = [add, multiply]

In [8]:
from pydantic import BaseModel, Field


# Note that the docstrings here are crucial, as they will be passed along
# to the model along with the class name.
class Add(BaseModel):
    """Add two integers together."""

    a: int = Field(..., description="First integer")
    b: int = Field(..., description="Second integer")


class Multiply(BaseModel):
    """Multiply two integers together."""

    a: int = Field(..., description="First integer")
    b: int = Field(..., description="Second integer")



In [17]:
from langchain_openai import ChatOpenAI
class PythonCode(BaseModel):
    """Multiply two integers together."""

    code: str = Field(..., description="Executable python code. should not include emoticons or other ono codeing texts")

tools = [PythonCode]
llm = ChatOpenAI(model="gpt-4.1")
llm_with_tools = llm.bind_tools(tools)

query = "Write the python code to print hello"

reps = llm_with_tools.invoke(query)

In [18]:
reps

AIMessage(content='Here is the Python code to print "hello":\n\n```python\nprint("hello")\n```', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 67, 'total_tokens': 86, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_422e2d36a8', 'id': 'chatcmpl-CP3z90CdphmVQdRV5qaX4pvnLST7T', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--5cfc2e84-2e30-43f0-9540-900ffb0a05ec-0', usage_metadata={'input_tokens': 67, 'output_tokens': 19, 'total_tokens': 86, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [19]:
reps.tool_calls

[]